In [1]:
import torch
import numpy as np
import scipy.stats as st
import sys
import os
sys.path.append("..")

from metamotivo.agents.fb.flow_bc.agent import FBFlowBCAgent
from metamotivo.envs.ogbench import OGBenchEnvConfig, ALL_TASKS
from metamotivo.data_loading.ogbench import OGBenchDataConfig
from metamotivo.envs.utils.rollout import rollout

def calculate_success(infos):
    """Computes binary success (True/False) for each episode in the rollout."""
    return [any([step.get("success", False) for step in info]) for info in infos]

def get_ogbench_config(results_path, domain, task, results_root="/home/jovyan/bobrin/td_jepa/results_fb_ogbench_proprio"):
    """Robustly find best OGBench params from your sweep file, falling back to disk search if missing."""
    import json
    from pathlib import Path
    
    ckpt_path = None
    best_params = None
    
    # 1. Try to load from sweep results
    if results_path and os.path.exists(results_path):
        try:
            with open(results_path, "r") as f:
                data = json.load(f)
                summary = data.get("summary", {})
                best_entry = None
            for entry_data in summary.values():
                if entry_data.get("domain") == domain and entry_data.get("task") == task:
                    # Handle both 'best_zol_sr' and the older 'best_zol_score'
                    score = entry_data.get("best_zol_sr", entry_data.get("best_zol_score", -1))
                    if best_entry is None or score > best_entry.get("best_zol_sr", best_entry.get("best_zol_score", -1)):
                        best_entry = entry_data
            
            if best_entry: 
                ckpt_path = best_entry.get("checkpoint")
                best_params = best_entry.get("best_zol_params")
        except (json.JSONDecodeError, IOError):
            pass
            
    # 2. Fallback to searching on disk if checkpoint not found
    if not ckpt_path:
        root = Path(results_root)
        domain_dir = root / domain
        if domain_dir.exists():
            for seed_dir in domain_dir.glob("*"):
                potential_ckpt = seed_dir / "checkpoint"
                if potential_ckpt.exists():
                    ckpt_path = str(potential_ckpt)
                    print(f"  Fallback: Found checkpoint on disk: {ckpt_path}")
                    break
                    
    return ckpt_path, best_params

def run_ogbench_zol_task_evaluation(agent, task_env, env_cfg, domain, task, batch, zol_search_params=None):
    """Performs baseline eval, ZOL search, and post-search eval with Success Rate reporting."""
    print(f"\n--- [OGBench Task: {task}] ---")
    device = agent.device
    
    if zol_search_params is None:
        zol_search_params = {
            "mu_source": "init",      # Better for navigation
            "use_exp_weights": True,
            "weight_temp": 1.0,       # Lower temp for stability
            "mu_reward_top_frac": 0.05,
            "self_normalized_obj": True,
        }
    
    # Use .detach() to avoid RuntimeError
    relabel_fn = env_cfg.get_relabel_fn(task)
    next_physics = batch["next"]["physics"].detach().cpu().numpy()
    actions = batch["action"].detach().cpu().numpy()
    rewards_np = relabel_fn(next_physics, actions)
    
    # CRITICAL: Reward Shift to [0, 1]
    rewards_np += 1.0
    
    rewards = torch.tensor(rewards_np, dtype=torch.float32).to(device)
    batch_obs = batch["next"]["observation"].to(device)
    
    # Baseline
    initial_z = agent._model.reward_inference(batch_obs, rewards.reshape(-1, 1))
    print(f"  Evaluating Baseline (100 episodes)...")
    base_stats, base_infos, _ = rollout(task_env, agent=agent._model, ctx=initial_z, num_episodes=100)
    base_successes = calculate_success(base_infos)
    base_sr_m, base_sr_ci = get_stats(base_successes)
    base_rew_m, base_rew_ci = get_stats(base_stats['reward'])
    print(f"  Baseline: {base_sr_m*100:.1f}% success ({base_rew_m:.2f} reward)")

    # ZOL optimization
    print(f"  Optimizing z (ZOL Search)...")
    z_zol = agent.zol_latent_search(task_env, batch_obs, rewards.flatten(), initial_z, **zol_search_params)
    
    # Final Eval
    print(f"  Evaluating ZOL (100 episodes)...")
    zol_stats, zol_infos, _ = rollout(task_env, agent=agent._model, ctx=z_zol, num_episodes=100)
    zol_successes = calculate_success(zol_infos)
    zol_sr_m, zol_sr_ci = get_stats(zol_successes)
    zol_rew_m, zol_rew_ci = get_stats(zol_stats['reward'])
    print(f"  ZOL:      {zol_sr_m*100:.1f}% success ({zol_rew_m:.2f} reward)")
    
    return {
        "base_sr": (base_sr_m, base_sr_ci),
        "zol_sr": (zol_sr_m, zol_sr_ci)
    }

def get_stats(data):
    mean = np.mean(data)
    sem = st.sem(data)
    ci = 1.96 * sem
    return mean, ci

In [2]:
# Configuration
OG_DOMAIN = "antmaze-large-navigate-v0" 
RESULTS_PATH = "../zol_sweep_ogbench_results.json"
DATASET_ROOT = "/home/jovyan/bobrin/td_jepa/ogbench_data"

# 1. Load Data for Reward Inference
data_cfg = OGBenchDataConfig(domain=OG_DOMAIN, dataset_root=DATASET_ROOT)
replay_buffer = data_cfg.build(buffer_device="cuda", batch_size=10_000, frame_stack=1)
batch = replay_buffer["train"].sample(10_000)

# 2. Iterate through tasks
all_results = {}
agent = None

for i, task in enumerate(ALL_TASKS[OG_DOMAIN]):
    env_cfg = OGBenchEnvConfig(domain=OG_DOMAIN, task=task)
    task_env, _ = env_cfg.build()
    
    if agent is None:
        ckpt_path, best_params = get_ogbench_config(RESULTS_PATH, OG_DOMAIN, task)
        if ckpt_path is None:
            raise ValueError(f"Could not find checkpoint for {OG_DOMAIN}. Check your RESULTS_PATH or disk.")
            
        agent = FBFlowBCAgent.load(
            ckpt_path, 
            device="cuda", 
            obs_space=task_env.observation_space, 
            action_dim=batch["action"].shape[-1]
        )
        agent._model.train(False)
    
    # Update agent config if we have best params from sweep
    search_kwargs = None
    if best_params:
        print(f"  Applying sweep params: {best_params}")
        config_keys = {"lr", "num_steps", "n_mu", "early_stop_patience", "early_stop_tol", 
                    "chi2_coef", "trust_l2_coef", "weight_clip", "center_rewards"}
        cfg_updates = {k: v for k, v in best_params.items() if k in config_keys}
        search_kwargs = {k: v for k, v in best_params.items() if k not in config_keys}
        
        if cfg_updates:
            agent.cfg = agent.cfg.model_copy(update={
                "train": agent.cfg.train.model_copy(
                    update={"zol": agent.cfg.train.zol.model_copy(update=cfg_updates)}
                )
            })
            
    res = run_ogbench_zol_task_evaluation(agent, task_env, env_cfg, OG_DOMAIN, task, batch, zol_search_params=search_kwargs)
    all_results[task] = res
    task_env.close()

# 3. Print Final Summary Table
print(f"\n{'Task':<30} | {'Baseline Success':<20} | {'ZOL Success':<20}")
print("-" * 75)
for task, res in all_results.items():
    b_m, b_ci = res["base_sr"]
    z_m, z_ci = res["zol_sr"]
    print(f"{task:<30} | {b_m*100:6.1f}% ± {b_ci*100:4.1f}%     | {z_m*100:6.1f}% ± {z_ci*100:4.1f}%")

Loading data from: /home/jovyan/bobrin/td_jepa/ogbench_data/antmaze-large-navigate-v0/buffer
  Fallback: Found checkpoint on disk: /home/jovyan/bobrin/td_jepa/results_fb_ogbench_proprio/antmaze-large-navigate-v0/1/checkpoint
compile True
compiling with mode 'reduce-overhead'
cudagraphs False

--- [OGBench Task: antmaze-large-navigate-singletask-task1-v0] ---
  Evaluating Baseline (100 episodes)...
  Baseline: 60.0% success (-786.76 reward)
  Optimizing z (ZOL Search)...
  Evaluating ZOL (100 episodes)...
  ZOL:      62.0% success (-776.51 reward)

--- [OGBench Task: antmaze-large-navigate-singletask-task2-v0] ---
  Evaluating Baseline (100 episodes)...
  Baseline: 81.0% success (-664.59 reward)
  Optimizing z (ZOL Search)...
  Evaluating ZOL (100 episodes)...
  ZOL:      82.0% success (-669.18 reward)

--- [OGBench Task: antmaze-large-navigate-singletask-task3-v0] ---
  Evaluating Baseline (100 episodes)...
  Baseline: 24.0% success (-904.72 reward)
  Optimizing z (ZOL Search)...
  Eva